In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import crosscoders as xc
import torch
# xc.dataclasses.configs.RunnerConfig(xc.dataclasses.configs.ModelConfig('acausal'))


------------------------- CONSTANTS -------------------------
GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/yandy/repos/crosscoders',
    CONFIG_FILEPATH = '/home/yandy/repos/crosscoders/src/scripts/configs/data.yml',
    DATA_DIR = '/home/ec2-user/crosscoders/data',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 32,
        MAX_RECORDS = None,
        MAX_BATCHES = 1000000,
        MAX_TOKENS = 10000000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)
-------------------------------------------------------------



In [4]:
from crosscoders import CONSTANTS

CONSTANTS.EXPERIMENT.MAX_TOKENS = 1000000

In [14]:
import datasets
import ray
from crosscoders.data.dataset import TinyStoriesRayDataset


# train_ds = TinyStoriesRayDataset().load('activations')
hf_dataset = datasets.load_dataset('roneneldan/TinyStories', streaming=True)
train_ds = ray.data.from_huggingface(hf_dataset['train'], concurrency=1)

In [15]:
train_dl = train_ds.iter_torch_batches(
    batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    # local_shuffle_buffer_size=16
    device='cuda'
)


# for batch_idx, batch in enumerate(train_dl):

#     # loss = runner.training_step(batch)

#     # print(loss)

#     break

In [3]:
from transformer_lens import HookedTransformer

tiny_stories_1m_model = HookedTransformer.from_pretrained('tiny-stories-1M')

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Loaded pretrained model tiny-stories-1M into HookedTransformer


In [147]:
batch = train_ds.take_batch(5)



# model_names = ('tiny-stories-1M', 'tiny-stories-3M') #, 'tiny-stories-8M', 'tiny-stories-28M'):
# model_names = ('pythia-160m', 'pythia-160m-deduped')
model_names = ('gpt2-small', 'gpt-neo-125M')
models = {
    mn: HookedTransformer.from_pretrained(mn)
    for mn in model_names
}

# TODO: verify tokenizers are the same or make them



2025-02-20 15:53:25,535	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-20_13-32-20_494581_8816/logs/ray-data
2025-02-20 15:53:25,535	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadHuggingFace] -> LimitOperator[limit=5]


Running 0: 0.00 row [00:00, ? row/s]

- ReadHuggingFace->SplitBlocks(15) 1: 0.00 row [00:00, ? row/s]

- limit=5 2: 0.00 row [00:00, ? row/s]

Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt-neo-125M into HookedTransformer


In [160]:

# def store_activation(out, mn, tensor, hook):
def store_activation_hook(activation, hook, model_name, latent_name, layer_idx):
# def store_activation_hook(activation, hook, layer_idx):

    # raise NotImplementedError(activation.shape)

    global out

    # layer_idx, latent_name = (lambda _: [int(_[1]), _[2][5:]])(hook.name.split('.'))

    out[f'{model_name}.{latent_name}'][...,layer_idx,:] = activation.detach()

In [54]:
from crosscoders.dataclasses.configs.globals import HardwareConfig


# layer_hooks = [
#     (f"blocks.{layer_idx}.hook_resid_post", store_resid_post_activation)
#     for layer_idx in range(8)
# ]

latent_names = ('resid_post',)



In [161]:

from functools import partial


tokenizer_model = next(iter(models.values()))

tokens = tokenizer_model.to_tokens(batch['text'].tolist())
out = {
    'tokens': tokens
}


for mn, model in models.items():


    # hooks = [
    #     model.blocks[layer_idx].__getattr__(f'hook_{ln}').register_forward_hook(
    #         lambda module, input, output: store_activation(out, mn, ln, layer_idx, input, output)
    #     )
    #     for ln in latent_names
    #     for layer_idx in range(model.cfg.n_layers)
    # ]


    # compose tensors for desired latent_names, add to latents dict
    for ln in latent_names:
        out[f'{mn}.{ln}'] = torch.empty(
            (*tokens.shape, model.cfg.n_layers, model.cfg.d_model),
            **HardwareConfig().asdict()
        )

    with torch.inference_mode():
        _ = model.run_with_hooks(
            tokens,
            fwd_hooks=[
                (f'blocks.{layer_idx}.hook_{ln}', partial(store_activation_hook, model_name=mn, latent_name=ln, layer_idx=layer_idx))
                for ln in latent_names
                for layer_idx in range(model.cfg.n_layers)
            ]
        )
        # _ = model(tokens)


    # for hook in hooks:
    #     hook.remove()




bos_token = tokenizer_model.to_single_token(tokenizer_model.tokenizer.bos_token)
col_indices = torch.arange(out['tokens'].shape[1]).unsqueeze(0).expand_as(out['tokens']).to(out['tokens'].device)
mask = (col_indices != 0) & (out['tokens'] != bos_token)

for k, v in out.items():
    out[k] = v[mask]

In [134]:
out.keys()

dict_keys(['tokens', 'gpt2-small.resid_post', 'gpt-neo-125M.resid_post'])

In [135]:
{k: v.shape for k, v in out.items()}

{'tokens': torch.Size([903]),
 'gpt2-small.resid_post': torch.Size([903, 12, 768]),
 'gpt-neo-125M.resid_post': torch.Size([903, 12, 768])}

In [164]:
from crosscoders.data.preprocessing import TokenToLatents


ttl = TokenToLatents()

Loaded pretrained model gpt2-small into HookedTransformer
Loaded pretrained model gpt-neo-125M into HookedTransformer


In [165]:
batch = ttl(batch)

In [166]:
{k: v.shape for k, v in out.items()}

{'tokens': torch.Size([903]),
 'gpt2-small.resid_post': torch.Size([903, 12, 768]),
 'gpt-neo-125M.resid_post': torch.Size([903, 12, 768])}

In [172]:
train_ds_ = train_ds.map_batches(
    TokenToLatents,
    batch_size=8,
    concurrency=1,
    num_gpus=1,
    # num_cpus=1
)

train_dl = train_ds_.iter_torch_batches(
    batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    # local_shuffle_buffer_size=16
    device='cuda'
)


In [173]:

for batch_idx, batch in enumerate(train_dl):

    # loss = runner.training_step(batch)

    # print(loss)

    break

2025-02-20 16:07:57,988	WARNING map_operator.py:701 -- Specifying both num_cpus and num_gpus for map tasks is experimental, and may result in scheduling or stability issues. Please report any issues to the Ray team: https://github.com/ray-project/ray/issues/new/choose


2025-02-20 16:07:57,991	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-20_13-32-20_494581_8816/logs/ray-data
2025-02-20 16:07:57,993	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadHuggingFace] -> ActorPoolMapOperator[MapBatches(TokenToLatents)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadHuggingFace->SplitBlocks(15) 1: 0.00 row [00:00, ? row/s]

- MapBatches(TokenToLatents) 2: 0.00 row [00:00, ? row/s]

In [175]:
{k: v.shape for k, v in batch.items()}

{'tokens': torch.Size([32]),
 'gpt2-small.resid_post': torch.Size([32, 12, 768]),
 'gpt-neo-125M.resid_post': torch.Size([32, 12, 768])}